# iTantra MT export -- IndicTrans2 to ONNX, on Colab

Runs `tools/export_indictrans2_onnx.py` + `tools/quantize_and_verify.py` from the `translation-mt` branch against the real IndicTrans2-Distilled checkpoints. This **must** run somewhere with real `huggingface.co` access and your own accepted license -- see `docs/CLAUDE.md`.

**Before running:** the checkpoints are gated. Log into huggingface.co in a browser and click *Agree and access repository* on both:
- https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M
- https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M

A token alone does not bypass this -- the click-through has to happen once per account.

**Runtime:** Runtime -> Change runtime type -> GPU (T4 is fine; export/quantize are quick, the checkpoints are 200M-parameter distilled models).

In [ ]:
# 1. Get the tooling. If the repo is private, Colab will prompt for GitHub credentials/a token.
!git clone --branch translation-mt --single-branch https://github.com/Naitik328/itantra.git
%cd itantra/tools

In [ ]:
# 2. Dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. HF auth -- paste a token with read access after accepting both licenses above.
from huggingface_hub import login
login()

## 4. Introspect first
Cheap sanity check before the real export -- confirms the checkpoint loads, prints vocab size / hidden size / tokenizer class. If `AutoTokenizer(trust_remote_code=True)` fails here, rerun with `--tokenizer-type indictrans-toolkit`.

In [ ]:
!python export_indictrans2_onnx.py --direction en-indic --out /content/mt/en-indic --introspect-only

## 5. en-indic direction: export, quantize, verify
`--embed-tokenizer` is on by default (bakes SentencePiece tokenization into the ONNX graph -- see `tools/tokenizer_graph.py`). `verify_tokenizer_ids()` inside `quantize_and_verify.py` is the step that actually confirms or refutes the `--add-bos`/`--add-eos`/`--fairseq-vocab-shift` defaults; **read its output, it hard-fails on a mismatch rather than warning.**

In [ ]:
!python export_indictrans2_onnx.py --direction en-indic --out /content/mt/en-indic

In [ ]:
!python quantize_and_verify.py \
  --in-dir /content/mt/en-indic --out-dir /content/mt/en-indic \
  --direction en-indic --src-lang en --tgt-lang hi \
  --test-file test_sentences/en-hi.tsv

If `verify_tokenizer_ids()` fails above: read the printed `in-graph` vs `reference` id lists, work out which flag explains the difference (a leading/trailing id that shouldn't be there usually means `--add-bos`/`--add-eos` is wrong; a systematic off-by-one across every id usually means `--fairseq-vocab-shift` needs to flip), then re-run the export cell with the corrected flag, e.g.:
```
!python export_indictrans2_onnx.py --direction en-indic --out /content/mt/en-indic --add-bos --fairseq-vocab-shift
```
and re-run verify until it passes clean. Do not proceed past a failing verify.

## 6. indic-en direction: export, quantize, verify
Whatever flags fixed en-indic above, try the same ones here first -- both checkpoints share the same export pipeline and were very likely tokenized the same way, but confirm independently rather than assuming.

In [ ]:
!python export_indictrans2_onnx.py --direction indic-en --out /content/mt/indic-en

In [ ]:
!python quantize_and_verify.py \
  --in-dir /content/mt/indic-en --out-dir /content/mt/indic-en \
  --direction indic-en --src-lang hi --tgt-lang en \
  --test-file test_sentences/hi-en.tsv

## 7. Report back
Copy the output of the next cell into the conversation (or the PR) -- sizes, and whether both `verify_tokenizer_ids()` runs passed clean or needed non-default flags. That result is what turns the "UNVERIFIED" notes in `tools/tokenizer_graph.py` / `translation/README.md` into confirmed facts (or a bug report if the in-graph approach turns out not to work for this checkpoint at all).

In [ ]:
import os
for direction in ("en-indic", "indic-en"):
    d = f"/content/mt/{direction}"
    print(f"== {direction} ==")
    for name in sorted(os.listdir(d)):
        print(f"  {name:30s} {os.path.getsize(os.path.join(d, name)) / 1e6:8.2f} MB")

## 8. Get the files out of Colab
Pick one. The shipped set per direction is `encoder.int8.onnx`, `decoder.int8.onnx`, `detokenizer.onnx`, `vocab_ids.json`, `SHA256SUMS.txt` (the `encoder.onnx`/`decoder.onnx`/`encoder_plain.onnx`/`tokenizer_bridge_debug.onnx` fp32 intermediates and debug graph don't need to leave Colab).

In [ ]:
# Option A: zip and download directly
!cd /content/mt && zip -r /content/mt_export.zip \
  */encoder.int8.onnx */decoder.int8.onnx */detokenizer.onnx */vocab_ids.json */SHA256SUMS.txt
from google.colab import files
files.download('/content/mt_export.zip')

In [ ]:
# Option B: copy to Google Drive instead (bigger files, flaky download)
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/itantra_mt_export
# !cp -r /content/mt/* /content/drive/MyDrive/itantra_mt_export/